# Predicting next-month churn
Question: can we predict `churn_next_month` from the monthly account snapshot?

Design decisions (following checklist.md):
- **Leakage:** `lifetime_months` comes from a CRM export of 2026-01-05, so it is only known after the account's life has played out. It is not available at prediction time and is dropped. `account_ref` is an identifier and is dropped.
- **Test set:** the model is meant for future months, so the test set is the latest months. Accounts repeat across rows, so test accounts are held out whole: any account with a row in the test window is removed from training entirely.
- **CV:** grouped by account (whole accounts per fold), inside the training rows only. Preprocessing lives in a Pipeline, so it is fit per fold.

In [ ]:
import pandas as pd, numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
df = pd.read_csv('data.csv')
print(df.shape, df.account_ref.nunique(), df.churn_next_month.mean())

## 1. Split first

In [ ]:
CUTOFF = '2025-09'   # test window: 2025-09 .. 2025-11 (latest rows)
test_rows = df.snapshot_month >= CUTOFF
test_accts = set(df.loc[test_rows, 'account_ref'])
test = df[test_rows].reset_index(drop=True)
train = df[~test_rows & ~df.account_ref.isin(test_accts)].reset_index(drop=True)
assert not set(train.account_ref) & set(test.account_ref)
assert train.snapshot_month.max() < test.snapshot_month.min()
print('train', train.shape, train.account_ref.nunique(), train.churn_next_month.mean())
print('test ', test.shape, test.account_ref.nunique(), test.churn_next_month.mean())

## 2. Features and candidate models (named before any scoring)
Features: plan, seats, monthly_fee, tenure_months, usage_hours, support_tickets.

Candidates:
1. `baseline`: predicts the training prevalence (trivial).
2. `logreg`: logistic regression, C in {0.1, 1, 10}.
3. `hgb`: gradient boosting, learning_rate 0.05, max_depth in {2, 3}.

Selection metric: average precision (classes are imbalanced); ROC-AUC reported too. The decision threshold is chosen on out-of-fold predictions to maximise F1.

In [ ]:
num = ['seats', 'monthly_fee', 'tenure_months', 'usage_hours', 'support_tickets']; cat = ['plan']
def pre(scale):
    return ColumnTransformer([('n', StandardScaler() if scale else 'passthrough', num),
                              ('c', OneHotEncoder(handle_unknown='ignore'), cat)])
def make(name, **p):
    if name == 'baseline':
        return Pipeline([('pre', pre(False)), ('m', DummyClassifier(strategy='prior'))])
    if name == 'logreg':
        return Pipeline([('pre', pre(True)), ('m', LogisticRegression(C=p['C'], max_iter=1000))])
    return Pipeline([('pre', pre(False)),
                     ('m', HistGradientBoostingClassifier(learning_rate=0.05, max_depth=p['d'], max_iter=200, random_state=0))])
cands = {'baseline': make('baseline'),
         **{f'logreg_C{c}': make('logreg', C=c) for c in (0.1, 1, 10)},
         **{f'hgb_d{d}': make('hgb', d=d) for d in (2, 3)}}
X = train[num + cat]; y = train.churn_next_month; g = train.account_ref
cv = GroupKFold(n_splits=5)
oof, rows = {}, []
for k, m in cands.items():
    p = cross_val_predict(m, X, y, groups=g, cv=cv, method='predict_proba')[:, 1]
    oof[k] = p
    rows.append((k, roc_auc_score(y, p), average_precision_score(y, p)))
cvres = pd.DataFrame(rows, columns=['model', 'cv_auc', 'cv_ap']).sort_values('cv_ap', ascending=False)
cvres

In [ ]:
best = cvres.iloc[0].model
ths = np.linspace(0.02, 0.8, 157)
f1s = [f1_score(y, oof[best] >= t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(thr, 3), '| CV F1:', round(max(f1s), 3))

## 3. Choice (fixed before touching the test set)
The model and threshold printed above were selected on grouped cross-validation only. The test set is scored once below.

In [ ]:
final = cands[best].fit(X, y)
Xt = test[num + cat]; yt = test.churn_next_month
pt = final.predict_proba(Xt)[:, 1]
print(f'HEADLINE test ROC-AUC {roc_auc_score(yt, pt):.3f}  PR-AUC {average_precision_score(yt, pt):.3f}  F1@thr {f1_score(yt, pt >= thr):.3f}')
print(f'no-skill reference: AUC 0.5, PR-AUC = prevalence = {yt.mean():.3f}')

## 4. Conclusion
Compare the headline test ROC-AUC/PR-AUC with the no-skill values. The test score was computed once, after the choice was fixed. Caveats: the test window is only 3 months, so the estimate is noisy; and `lifetime_months` was dropped deliberately because it encodes the outcome.